In [ ]:
import cdsapi
import xarray as xr
import numpy as np
from pathlib import Path
import json
from datetime import datetime

class ERA5_6Hourly_Downloader:
    def __init__(self, output_dir="./era5_6hourly"):
        self.client = cdsapi.Client()
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
        # Europe bounds [North, West, South, East]
        self.europe_bounds = [75, -25, 30, 50]
        
        # 6-hourly timesteps (4 per day)
        self.times_6hourly = ['00:00', '06:00', '12:00', '18:00']
        
        # Essential variables for precipitation forecasting
        self.variable_sets = self.get_precipitation_variables()
        
    def get_precipitation_variables(self):
        """Optimized variable sets for precipitation forecasting"""
        
        return {
            # Core precipitation variables (Priority 1)
            'core_precipitation': [
                'total_precipitation',                    # Main target
                'convective_precipitation',               # Convective component
                'large_scale_precipitation',              # Stratiform component
                'total_column_water_vapour',              # Atmospheric moisture
            ],
            
            # Atmospheric dynamics (Priority 1)
            'dynamics': [
                '2m_temperature',                         # Surface temperature
                'surface_pressure',                       # Surface pressure
                'mean_sea_level_pressure',                # Sea level pressure
                '10m_u_component_of_wind',                # Zonal wind
                '10m_v_component_of_wind',                # Meridional wind
            ],
            
            # Moisture and clouds (Priority 2)
            'moisture_clouds': [
                '2m_dewpoint_temperature',                # Dewpoint
                'relative_humidity',                      # Relative humidity
                'total_cloud_cover',                      # Total clouds
                'low_cloud_cover',                        # Low clouds
                'high_cloud_cover',                       # High clouds
            ],
            
            # Atmospheric instability (Priority 2)
            'instability': [
                'convective_available_potential_energy',  # CAPE
                'convective_inhibition',                  # CIN
                'k_index',                                # K-index
                'lifted_index',                           # Lifted index
            ],
            
            # Energy and fluxes (Priority 3)
            'energy_fluxes': [
                'surface_sensible_heat_flux',             # Sensible heat
                'surface_latent_heat_flux',               # Latent heat
                'surface_solar_radiation_downwards',      # Solar radiation
                'surface_net_solar_radiation',            # Net solar
            ],
            
            # Moisture transport (Priority 3)
            'moisture_transport': [
                'vertical_integral_of_divergence_of_moisture_flux',     # Moisture convergence
                'vertical_integral_of_eastward_water_vapour_flux',      # Eastward moisture flux
                'vertical_integral_of_northward_water_vapour_flux',     # Northward moisture flux
            ],
            
            # Soil and surface (Priority 4)
            'soil_surface': [
                'skin_temperature',                       # Surface temperature
                'volumetric_soil_water_layer_1',          # Soil moisture
                'snow_depth',                             # Snow depth
                'soil_temperature_level_1',               # Soil temperature
            ]
        }
    
    def download_by_priority(self, 
                           priorities=[1, 2], 
                           start_year=1940, 
                           end_year=2024,
                           chunk_years=10):
        """Download variables by priority levels"""
        
        priority_mapping = {
            1: ['core_precipitation', 'dynamics'],
            2: ['moisture_clouds', 'instability'], 
            3: ['energy_fluxes', 'moisture_transport'],
            4: ['soil_surface']
        }
        
        # Get variable sets for selected priorities
        selected_sets = []
        for priority in priorities:
            selected_sets.extend(priority_mapping.get(priority, []))
        
        print(f"Downloading priority levels: {priorities}")
        print(f"Variable sets: {selected_sets}")
        
        downloaded_files = []
        
        # Download each variable set
        for var_set_name in selected_sets:
            variables = self.variable_sets[var_set_name]
            
            print(f"\n{'='*50}")
            print(f"Downloading: {var_set_name.upper()}")
            print(f"Variables ({len(variables)}): {variables}")
            print(f"{'='*50}")
            
            # Download in year chunks
            year_chunks = self.create_year_chunks(start_year, end_year, chunk_years)
            
            for chunk_idx, years in enumerate(year_chunks):
                filename = f"era5_{var_set_name}_{min(years)}_{max(years)}_europe_6h.nc"
                filepath = self.output_dir / filename
                
                if filepath.exists():
                    print(f"✓ EXISTS: {filename}")
                    downloaded_files.append(filepath)
                    continue
                
                print(f"\nDownloading chunk {chunk_idx+1}/{len(year_chunks)}: {filename}")
                print(f"Years: {min(years)}-{max(years)}")
                
                try:
                    self.client.retrieve(
                        'reanalysis-era5-single-levels',
                        {
                            'product_type': 'reanalysis',
                            'variable': variables,
                            'year': [str(y) for y in years],
                            'month': [f'{m:02d}' for m in range(1, 13)],
                            'day': [f'{d:02d}' for d in range(1, 32)],
                            'time': self.times_6hourly,  # 6-hourly timestamps
                            'area': self.europe_bounds,
                            'format': 'netcdf',
                            'grid': [0.25, 0.25],  # 0.25° resolution
                        },
                        str(filepath)
                    )
                    
                    # Verify download
                    self.verify_download(filepath)
                    downloaded_files.append(filepath)
                    
                except Exception as e:
                    print(f"✗ ERROR: {e}")
                    continue
        
        return downloaded_files
    
    def verify_download(self, filepath):
        """Verify downloaded file"""
        try:
            ds = xr.open_dataset(filepath)
            
            print(f"  ✓ Shape: {dict(ds.dims)}")
            print(f"  ✓ Variables: {len(ds.data_vars)}")
            print(f"  ✓ Time range: {ds.time.values[0]} to {ds.time.values[-1]}")
            print(f"  ✓ File size: {filepath.stat().st_size / (1024**3):.2f} GB")
            
            # Check for missing values
            for var in ds.data_vars:
                missing_pct = (ds[var].isnull().sum() / ds[var].size * 100).values
                if missing_pct > 5:
                    print(f"  ⚠ Warning: {var} has {missing_pct:.1f}% missing values")
            
            ds.close()
            
        except Exception as e:
            print(f"  ✗ Verification failed: {e}")
    
    def create_year_chunks(self, start_year, end_year, chunk_size):
        """Split years into manageable chunks"""
        years = list(range(start_year, end_year + 1))
        return [years[i:i + chunk_size] for i in range(0, len(years), chunk_size)]
    
    def download_static_fields(self):
        """Download time-invariant fields"""
        
        static_file = self.output_dir / "era5_static_europe.nc"
        
        if static_file.exists():
            print("✓ Static fields already exist")
            return static_file
        
        print("Downloading static fields...")
        
        static_variables = [
            'geopotential',           # Terrain height
            'land_sea_mask',          # Land-sea mask
            'soil_type',              # Soil type
            'lake_cover',             # Lake cover
            'low_vegetation_cover',   # Low vegetation
            'high_vegetation_cover',  # High vegetation
        ]
        
        try:
            self.client.retrieve(
                'reanalysis-era5-single-levels',
                {
                    'product_type': 'reanalysis',
                    'variable': static_variables,
                    'year': '2020',
                    'month': '01',
                    'day': '01',
                    'time': '00:00',
                    'area': self.europe_bounds,
                    'format': 'netcdf',
                    'grid': [0.25, 0.25],
                },
                str(static_file)
            )
            
            self.verify_download(static_file)
            return static_file
            
        except Exception as e:
            print(f"✗ Static fields error: {e}")
            return None
    
    def merge_downloaded_files(self, files_by_category):
        """Merge files by category and create final dataset"""
        
        merged_files = {}
        
        for category, files in files_by_category.items():
            if not files:
                continue
                
            merged_file = self.output_dir / f"era5_{category}_merged_6h.nc"
            
            if merged_file.exists():
                print(f"✓ Merged file exists: {merged_file.name}")
                merged_files[category] = merged_file
                continue
            
            print(f"Merging {category} files...")
            
            try:
                # Load and concatenate files
                datasets = []
                for file in sorted(files):
                    if file.exists():
                        ds = xr.open_dataset(file)
                        datasets.append(ds)
                
                if datasets:
                    merged_ds = xr.concat(datasets, dim='time')
                    merged_ds = merged_ds.sortby('time')
                    
                    # Save merged dataset
                    merged_ds.to_netcdf(merged_file)
                    merged_files[category] = merged_file
                    
                    print(f"✓ Merged {len(datasets)} files into {merged_file.name}")
                    
                    # Close datasets
                    for ds in datasets:
                        ds.close()
                    merged_ds.close()
                
            except Exception as e:
                print(f"✗ Merge error for {category}: {e}")
        
        return merged_files
    
    def create_final_dataset(self, merged_files):
        """Combine all categories into single dataset for MESA-Net"""
        
        final_file = self.output_dir / "era5_precipitation_mesa_6h.nc"
        
        if final_file.exists():
            print(f"✓ Final dataset exists: {final_file}")
            return final_file
        
        print("Creating final combined dataset...")
        
        try:
            all_datasets = []
            variable_names = []
            
            for category, file in merged_files.items():
                ds = xr.open_dataset(file)
                
                # Add each variable as separate dataset
                for var_name in ds.data_vars:
                    all_datasets.append(ds[var_name])
                    variable_names.append(f"{category}_{var_name}")
            
            # Combine all variables
            combined_ds = xr.concat(all_datasets, dim='variable')
            combined_ds = combined_ds.assign_coords(variable=variable_names)
            
            # Save final dataset
            combined_ds.to_netcdf(final_file)
            
            print(f"✓ Final dataset created: {final_file}")
            print(f"  Shape: {combined_ds.shape}")
            print(f"  Variables: {len(variable_names)}")
            
            return final_file
            
        except Exception as e:
            print(f"✗ Final dataset creation failed: {e}")
            return None

# ============ USAGE FUNCTIONS ============

def download_essential_for_mesa():
    """Download essential variables for MESA-Net development"""
    
    downloader = ERA5_6Hourly_Downloader()
    
    print("ERA5 6-Hourly Download for MESA-Net")
    print("==================================")
    print("Target: Precipitation forecasting over Europe")
    print("Resolution: 0.25° spatial, 6-hourly temporal")
    print("Period: 1940-2024 (full ERA5)")
    print("")
    
    # Download Priority 1 & 2 variables (most important)
    files = downloader.download_by_priority(
        priorities=[1, 2],  # Core + important variables
        start_year=1940,
        end_year=2024,
        chunk_years=10
    )
    
    # Download static fields
    static_file = downloader.download_static_fields()
    
    # Organize files by category
    files_by_category = {}
    for file in files:
        category = file.name.split('_')[1]  # Extract category from filename
        if category not in files_by_category:
            files_by_category[category] = []
        files_by_category[category].append(file)
    
    # Merge files by category
    merged_files = downloader.merge_downloaded_files(files_by_category)
    
    # Create final dataset
    final_dataset = downloader.create_final_dataset(merged_files)
    
    # Summary
    print(f"\n{'='*50}")
    print("DOWNLOAD COMPLETE")
    print(f"{'='*50}")
    print(f"Downloaded files: {len(files)}")
    print(f"Merged categories: {len(merged_files)}")
    print(f"Final dataset: {final_dataset}")
    print(f"Ready for MESA-Net development!")
    
    return final_dataset, merged_files, static_file

def download_minimal_test():
    """Quick test download (recent years only)"""
    
    downloader = ERA5_6Hourly_Downloader()
    
    files = downloader.download_by_priority(
        priorities=[1],  # Only core variables
        start_year=2022,
        end_year=2024,
        chunk_years=5
    )
    
    return files

def download_comprehensive():
    """Full download with all priority levels"""
    
    downloader = ERA5_6Hourly_Downloader()
    
    files = downloader.download_by_priority(
        priorities=[1, 2, 3, 4],  # All variables
        start_year=1940,
        end_year=2024,
        chunk_years=10
    )
    
    static_file = downloader.download_static_fields()
    
    return files, static_file

# ============ MAIN EXECUTION ============

if __name__ == "__main__":
    # Choose your download strategy:
    
    print("Starting ERA5 6-hourly download...")
    
    # Recommended: Essential variables for development
    final_dataset, merged_files, static_file = download_essential_for_mesa()
    
    # Alternative: Quick test (uncomment if you want to test first)
    # files = download_minimal_test()
    
    # Alternative: Comprehensive download (uncomment if you want everything)
    # files, static_file = download_comprehensive()